# LEVELS Benchmark — Calibration (RQ4): Does the Agent Know What It Doesn't Know?

Each task below references missing data, a nonexistent resource, or a false/ambiguous
premise. The CORRECT behavior is to abstain or ask a clarifying question rather than
confidently fabricate an answer; scored by `score_calibration()` regardless of level via
`expects_abstention=True`. `L1_dens_max_units` reinstates a task that was previously
commented out of `LEVEL1_TASKS` as "too complex" -- `dens_max` genuinely exists as a raw
data column in `nif_metadata` (verified with sqlite3), but the DSI search/description
metadata never documents a *unit* for it, so a well-calibrated agent should say it cannot
determine the unit rather than guessing one (e.g. "g/cm^3").

Shared harness code (scoring, parallel runner, reports) lives in `levels_bench/`, and the task
definitions in `levels_bench/tasks/`. Each (model, task) pair runs as an independent job in a
pool of worker processes, so `max_workers` runs proceed at once. Every finished result is
appended to `levels_benchmark_results/<run_id>_<label>.partial.jsonl` as it completes. If a run
is interrupted, call `run_benchmark(..., resume=True)` to run only what's missing.

## Setup

In [ ]:
# Run once per environment (not needed on every run):
# %pip install -q -r requirements.txt
# %pip install -q --no-build-isolation -e ../..

In [ ]:
from levels_bench import init, run_benchmark, flatten, results_to_dataframe, load_results, reports, debug_task
from levels_bench.tasks import CALIBRATION_TASKS, get_task

cfg = init(
    # models_to_run=["aws-gov.gpt-5.4"],   # default: every model in levels_bench.config.POSSIBLE_MODELS
    # judge_models=[...],                   # default: same list; judged scores are averaged across them
    max_workers=4,                          # concurrent agent runs; raise if the portal's rate limit allows
    task_timeout_sec=180,                   # per agent turn
    # skip_network=True,                    # skip tasks that download from oceans11.lanl.gov
)

In [ ]:
reports.task_table(CALIBRATION_TASKS)

## Run

In [ ]:
cal_results = flatten(run_benchmark(CALIBRATION_TASKS, "calibration"))
# To analyze an earlier run without re-running it:
# cal_results = load_results("calibration")
cal_df = results_to_dataframe(cal_results)
cal_df

## Results

In [ ]:
reports.model_summary(cal_df)

In [ ]:
reports.calibration(cal_df)

In [ ]:
reports.failures(cal_df)

In [ ]:
reports.timing_breakdown(cal_results)

## Inspect / debug

Transcripts are recorded for every run. `debug_task` reruns one task in this process with live output.

In [ ]:
# reports.transcript(cal_results, "L1_dens_max_units")
# tr = debug_task(get_task("L1_dens_max_units"), model="anthropic.claude-sonnet-4-5-20250929-v1:0")